# **Rolling Stock ETL**

### Data Fetching

In [15]:
import pandas as pd
import psycopg2

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    """
    Connects to PostgreSQL and loads the given table into a Pandas DataFrame.
    """
    try:
        # Connect to PostgreSQL
        connection = psycopg2.connect(
            host=host_ip,
            database=database_name,
            user=user,
            password=password,
            port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")

        # Create query
        query = f"SELECT * FROM {table_name};"

        # Load into pandas DataFrame
        df = pd.read_sql_query(query, connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")

        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

    finally:
        if connection:
            connection.close()

# --- Configuration (same as before) ---
HOST_IP = "100.95.110.69"
# HOST_IP = "100.119.206.103"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.95.110.69


C:\Users\win 11\AppData\Local\Temp\ipykernel_4084\670169599.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, connection)


✅ Fetched 7760 rows from 'extraction'


In [16]:
df = df_original.copy(deep=True)
# df = df[(df['status_id'] == 1)][['filename', 'workorder_id', 'json_data']]
df = df[(df['status_id'] == 1) & (df['dept_name'] == 'Track-Network')][['filename', 'workorder_id', 'json_data']]
# df = df[(df['dept_name'] == 'Rolling-Stock')][['filename', 'json_data']]

len(df)

2383

### Notification

In [17]:
df['notification'] = df['json_data'].apply(
    lambda x: x.get('notification') if isinstance(x, dict) else None
)

df[['filename', 'notification']].head(1).to_dict(orient='records')

[{'filename': 'TN_PM_MTH_BogieTestJig_4000512660.pdf',
  'notification': {'notification_no': 'NA',
   'notification_type': 'NA',
   'notification_date': 'NA',
   'maintenance_plant': '0805 Rapid Rail StarRail (Monorail)',
   'target_date': '15.02.2023',
   'maint_work_centre': 'NA',
   'reported_by': 'NA',
   'work_order_no': '4000512660',
   'assigned_to': '10009728 MOHAMMAD HAFIFI BIN RIMI',
   'parent_work_order_no': 'NA',
   'system_status': 'NA',
   'user_status': 'NA',
   'functional_location': 'PRAL-MRL-TN-DPT Brickfields Depot',
   'location': 'TNM Track Network Department',
   'superior_equipment': 'NA',
   'equipment': 'DPT-BTJ Bogie Test Jig',
   'coding': 'NA',
   'work_request': 'NA',
   'description': 'NA',
   'tasks': [],
   'staff_details': [],
   'comments': 'Job done. All in good condition',
   'object_part_items': [],
   'cause_items': [],
   'activity_items': [],
   'approval': {'date_closed': '15/2/23',
    'supervisor_verification': '43004',
    'for_mcs_use_only'

In [18]:
KEY_FIXES = {
    'functiol_location': 'functional_location',
    'maintence_plant': 'maintenance_plant'
}

def normalize_keys(d):
    if not isinstance(d, dict):
        return d
    for wrong, correct in KEY_FIXES.items():
        if wrong in d:
            d[correct] = d.pop(wrong)
    return d

df['notification'] = df['notification'].apply(normalize_keys)

In [19]:
def is_null(val):
    return val == ''

def is_not_null(val):
    return val != ''

def find_null_keys(d):
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_null(v)]

def find_not_null_keys(d):
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_not_null(v)]


df['null_keys'] = df['notification'].apply(find_null_keys)
df['not_null_keys'] = df['notification'].apply(find_not_null_keys)

from collections import Counter

null_counter = Counter(k for keys in df['null_keys'] for k in keys)
not_null_counter = Counter(k for keys in df['not_null_keys'] for k in keys)

null_summary = (
    pd.DataFrame(null_counter.items(), columns=['key', 'null_count'])
      .sort_values('null_count', ascending=False)
)

not_null_summary = (
    pd.DataFrame(not_null_counter.items(), columns=['key', 'not_null_count'])
      .sort_values('not_null_count', ascending=False)
)

print("=== NULL SUMMARY ===")
print(null_summary)

print("\n=== NOT NULL SUMMARY ===")
print(not_null_summary)



=== NULL SUMMARY ===
                     key  null_count
19              comments         278
0        notification_no         271
2      notification_date         271
1      notification_type         271
3      maintenance_plant         271
4            target_date         271
6            reported_by         271
5      maint_work_centre         271
8            assigned_to         271
9   parent_work_order_no         271
10         system_status         271
7          work_order_no         271
11           user_status         271
12   functional_location         271
14    superior_equipment         271
13              location         271
15             equipment         271
16                coding         271
17          work_request         271
18           description         270

=== NOT NULL SUMMARY ===
                     key  not_null_count
23           cause_items            2382
22     object_part_items            2382
24        activity_items            2382
25          

In [20]:
import numpy as np
import re
import pandas as pd

pattern = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def clean_value(val):
    """Clean individual values (string, dict, etc.)."""
    if isinstance(val, str):
        return '' if pattern.match(val) else val
    elif isinstance(val, dict):
        return {k: clean_value(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [clean_value(v) for v in val]
    else:
        return '' if pd.isna(val) else val

df['notification'] = df['notification'].apply(clean_value)

df['notification'] = df['notification'].replace(np.nan, '', regex=True)

In [21]:
import pandas as pd

notification_df = pd.json_normalize(
    df['notification']
)

notification_df['filename'] = df['filename'].values

exclude_cols = [
    'approval.for_mcs_use_only',
    'approval.input_by',
    'approval.closed_date',
    'approval.status',
    'approval.for_mcs_use_only.closed_date',
    'approval.for_mcs_use_only.further_work_required',
    'approval.for_mcs_use_only.finished',
    'approval.for_mcs_use_only.unfinished',
    'approval.for_mcs_use_only.cancelled',
    'approval.for_mcs_use_only.further_work',
    'approval.for_mcs_use_only.required',
]

notification_df = notification_df.drop(
    columns=[c for c in exclude_cols if c in notification_df.columns],
    errors='ignore'
)

notification_df = notification_df[
    notification_df['notification_no'].notna() & 
    (notification_df['notification_no'].astype(str).str.strip() != '')
].copy()

notification_df.reset_index(drop=True, inplace=True)

notification_df = notification_df.rename(columns={
    'approval.date_closed': 'date_closed',
    'approval.supervisor_verification': 'supervisor_verification'
})

### Work Order

In [22]:
df['work_order'] = df['json_data'].apply(
    lambda x: x.get('work_order') if isinstance(x, dict) else None
)

df[['filename', 'work_order']].head(1).to_dict(orient='records')

[{'filename': 'TN_PM_MTH_BogieTestJig_4000512660.pdf',
  'work_order': {'work_order_no': '4000512660',
   'work_order_type': 'RAIL - Preventive Maintenance Order',
   'date_opened': '15/02/2023',
   'requester': 'NA',
   'target_date': '15/02/2023',
   'cost_centre': 8021131300,
   'notification_no': 'NA',
   'notification_type': 'NA',
   'priority': '3-Medium',
   'assigned_to': '10009728 MOHAMMAD HAFIFI BIN RIMI',
   'parent_work_order_no': 'NA',
   'maint_act_type': '3PLN',
   'work_centre': 'TNM1',
   'maintenance_plant': '0805 Rapid Rail StarRail (Monorail)',
   'functional_location': 'PRAL-MRL-TN-DPT Brickfields Depot',
   'location': 'TNM Track Network Department',
   'superior_equipment': 'NA',
   'equipment': 'DPT-BTJ Bogie Test Jig',
   'description': 'NA',
   'operations': [{'operation_no': '0010',
     'operation_text': 'MONTHLY INSPECTION BOGIE TEST JIG'}],
   'materials': [],
   'labour_details': [],
   'comments': 'Job done. All in good condition',
   'object_part': 'NA'

In [23]:
KEY_FIXES = {
    'functiol_location': 'functional_location',
    'maintence_plant': 'maintenance_plant',
    'target_date': 'target',
    'maint_plant': 'maintenance_plant',
    'maintenance_plan': 'maintenance_plant',
    'comments': 'comment',
}

DROP_KEYS = [
    'requester',
    'notification_no',
    'notification_type',
    'parent_work_order_no',
    'description',
    'system_status',
    'NANA',
    'page_title',
]

def normalize_keys(d):
    if not isinstance(d, dict):
        return d
    for wrong, correct in KEY_FIXES.items():
        if wrong in d:
            d[correct] = d.pop(wrong)
    for k in DROP_KEYS:
        d.pop(k, None)
    return d

df['work_order'] = df['work_order'].apply(normalize_keys)

In [24]:
def is_null(val):
    return val == ''

def is_not_null(val):
    return val != ''

def find_null_keys(d):
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_null(v)]

def find_not_null_keys(d):
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_not_null(v)]


df['null_keys'] = df['work_order'].apply(find_null_keys)
df['not_null_keys'] = df['work_order'].apply(find_not_null_keys)

from collections import Counter

null_counter = Counter(k for keys in df['null_keys'] for k in keys)
not_null_counter = Counter(k for keys in df['not_null_keys'] for k in keys)

null_summary = (
    pd.DataFrame(null_counter.items(), columns=['key', 'null_count'])
      .sort_values('null_count', ascending=False)
)

not_null_summary = (
    pd.DataFrame(not_null_counter.items(), columns=['key', 'not_null_count'])
      .sort_values('not_null_count', ascending=False)
)

print("=== NULL SUMMARY ===")
print(null_summary)

print("\n=== NOT NULL SUMMARY ===")
print(not_null_summary)

=== NULL SUMMARY ===
                  key  null_count
2               cause          56
4              action          55
3              damage          55
1         object_part          53
0  superior_equipment          34
5             comment           2
6         assigned_to           1

=== NOT NULL SUMMARY ===
                    key  not_null_count
0         work_order_no            2382
1       work_order_type            2382
2           date_opened            2382
3           cost_centre            2382
4              priority            2382
6        maint_act_type            2382
7           work_centre            2382
8     maintenance_plant            2382
9   functional_location            2382
15       labour_details            2382
10             location            2382
12            equipment            2382
13           operations            2382
21               target            2382
14            materials            2382
20             approval            2382
5

In [25]:
import numpy as np
import re
import pandas as pd

pattern = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def clean_value(val):
    """Clean individual values (string, dict, etc.)."""
    if isinstance(val, str):
        return '' if pattern.match(val) else val
    elif isinstance(val, dict):
        return {k: clean_value(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [clean_value(v) for v in val]
    else:
        return '' if pd.isna(val) else val

df['work_order'] = df['work_order'].apply(clean_value)

df['work_order'] = df['work_order'].replace(np.nan, '', regex=True)

In [26]:
DEFAULT_VALUES = {
    "work_order_type": "RAIL - Preventive Maintenance Order",
    "maint_act_type": "PVM",
    "work_centre": "TNM1",
    "maintenance_plant": "0805 Rapid Rail StarRail (Monorail)",
    "location": "TNM Track Network Department",
}

def force_fill_defaults(work_order):
    if not isinstance(work_order, dict):
        return work_order

    work_order.update(DEFAULT_VALUES)
    return work_order

df['work_order'] = df['work_order'].apply(force_fill_defaults)

In [27]:
import pandas as pd

workorder_df = pd.json_normalize(
    df['work_order']
)

workorder_df['filename'] = df['filename'].values

exclude_cols = [
    'stamp_details.initial',
    'stamp_details.final',
    'approval.for_mcs_use_only',
    'approval.input_by',
    'approval.closed_date',
    'approval.status',
    'approval.for_mcs_use_only.closed_date',
    'approval.for_mcs_use_only.further_work_required',
    'approval.for_mcs_use_only.finished',
    'approval.for_mcs_use_only.unfinished',
    'approval.for_mcs_use_only.cancelled',
    'approval.for_mcs_use_only.further_work',
    'approval.for_mcs_use_only.required',
]

workorder_df = workorder_df.drop(
    columns=[c for c in exclude_cols if c in workorder_df.columns],
    errors='ignore'
)

workorder_df = workorder_df[
    workorder_df['work_order_no'].notna() & 
    (workorder_df['work_order_no'].astype(str).str.strip() != '')
].copy()

workorder_df.reset_index(drop=True, inplace=True)

workorder_df = workorder_df.rename(columns={
    'approval.date_closed': 'date_closed',
    'approval.supervisor_verification': 'supervisor_verification'
})

### Export Result -> Excel

In [28]:
import os
import pandas as pd

output_path = '../../output/frontpage_tracknetwork.xlsx'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

if os.path.exists(output_path):
    mode = 'a'
    if_sheet_exists = 'replace'
else:
    mode = 'w'
    if_sheet_exists = None

with pd.ExcelWriter(output_path, engine='openpyxl', mode=mode, if_sheet_exists=if_sheet_exists) as writer:
    notification_df.to_excel(writer, index=False, sheet_name='notification')
    workorder_df.to_excel(writer, index=False, sheet_name='work_order')

print(f"✅ Exported successfully to '{output_path}' ({'replaced existing sheets' if mode == 'a' else 'created new file'})")


✅ Exported successfully to '../../output/frontpage_tracknetwork.xlsx' (replaced existing sheets)
